# GEMM Reference Model

Naive Python implementation of C = A × B for the Custom Vitis IP project.
Operands are `int16`, accumulator is `int32` — matches DSP48 arithmetic on Zynq-7000.

This notebook is the golden reference. The hardware will be verified against this output later.

NumPy is the only dependency. Constants for matrix dimensions live at the top so they're easy to change. Later these values will become AXI-Lite register writes when the hardware reads its config.

In [1]:
import numpy as np

M, N, K = 32, 32, 32

In [8]:
def gemm_naive(A, B):
    assert A.dtype == np.int16, f"A must be int16, got {A.dtype}"
    assert B.dtype == np.int16, f"B must be int16, got {B.dtype}"
    
    m, k = A.shape
    k2, n = B.shape
    assert k == k2, "inner dimensions must match"
    
    C = np.zeros((m, n), dtype=np.int32)
    
    for i in range(m):
        for j in range(n):
            acc = np.int32(0)
            for kk in range(k):
                acc += np.int32(A[i, kk]) * np.int32(B[kk, j])
            C[i, j] = acc
    
    return C

In [9]:
A = np.arange(1, M*K + 1, dtype=np.int16).reshape(M, K)
B = np.ones((K, N), dtype=np.int16)

C = gemm_naive(A, B)

print(A)
print(B)


[[   1    2    3 ...   30   31   32]
 [  33   34   35 ...   62   63   64]
 [  65   66   67 ...   94   95   96]
 ...
 [ 929  930  931 ...  958  959  960]
 [ 961  962  963 ...  990  991  992]
 [ 993  994  995 ... 1022 1023 1024]]
[[1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]
 ...
 [1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]]


In [10]:
C = gemm_naive(A, B)

print(f"C[0][0] = {C[0, 0]} (expected 528)")
print(f"C[0][1] = {C[0, 1]} (expected 528)")
print(f"C[1][0] = {C[1, 0]} (expected 1552)")

C[0][0] = 528 (expected 528)
C[0][1] = 528 (expected 528)
C[1][0] = 1552 (expected 1552)


## cross-check against NumPy's matmul

In [11]:
C_ref = A.astype(np.int32) @ B.astype(np.int32)

assert np.array_equal(C, C_ref), "naive loop disagrees with np.matmul"
print("naive GEMM matches np.matmul")

naive GEMM matches np.matmul


# tiled reference

In [12]:
def gemm_tiled(A, B, TM=8, TN=8, TK=8):
    assert A.dtype == np.int16, f"A must be int16, got {A.dtype}"
    assert B.dtype == np.int16, f"B must be int16, got {B.dtype}"

    m, k = A.shape
    k2, n = B.shape
    assert k == k2, "inner dimensions must match"
    assert m % TM == 0, f"M={m} must be divisible by TM={TM}"
    assert n % TN == 0, f"N={n} must be divisible by TN={TN}"
    assert k % TK == 0, f"K={k} must be divisible by TK={TK}"

    C = np.zeros((m, n), dtype=np.int32)

    for i_tile in range(m // TM):
        for j_tile in range(n // TN):
            for k_tile in range(k // TK):
                # Slice indices for this tile
                i_lo, i_hi = i_tile * TM, (i_tile + 1) * TM
                j_lo, j_hi = j_tile * TN, (j_tile + 1) * TN
                k_lo, k_hi = k_tile * TK, (k_tile + 1) * TK

                # Pull the input tiles
                A_tile = A[i_lo:i_hi, k_lo:k_hi]   # shape (TM, TK)
                B_tile = B[k_lo:k_hi, j_lo:j_hi]   # shape (TK, TN)

                # Tile multiply with int32 accumulation, accumulate into C
                C[i_lo:i_hi, j_lo:j_hi] += (
                    A_tile.astype(np.int32) @ B_tile.astype(np.int32)
                )

    return C

In [13]:
TM, TN, TK = 8, 8, 8
C_tiled = gemm_tiled(A, B, TM, TN, TK)

assert np.array_equal(C_tiled, C), \
    f"tiled disagrees with naive! max diff = {np.abs(C_tiled.astype(np.int64) - C.astype(np.int64)).max()}"

print(f"tiled GEMM matches naive (TM={TM}, TN={TN}, TK={TK})")
print(f"  C_tiled[0,0] = {C_tiled[0, 0]} (expected 528)")
print(f"  C_tiled[1,0] = {C_tiled[1, 0]} (expected 1552)")

tiled GEMM matches naive (TM=8, TN=8, TK=8)
  C_tiled[0,0] = 528 (expected 528)
  C_tiled[1,0] = 1552 (expected 1552)


# random data + tile-size sweep

In [14]:
# Reproducible random seed
np.random.seed(42)

# Random int16 inputs, bounded so the int32 accumulator stays safe
# (worst case: 32 * 100 * 100 = 320,000, well inside int32 range)
A_rand = np.random.randint(-100, 101, size=(M, K), dtype=np.int16)
B_rand = np.random.randint(-100, 101, size=(K, N), dtype=np.int16)

# Reference: trust gemm_naive from step 1.3
C_ref = gemm_naive(A_rand, B_rand)

# Sweep tile sizes
tile_sizes = [
    (4, 4, 4),
    (8, 8, 8),
    (16, 16, 16),
    (4, 8, 16),     # asymmetric — different tile size in each dimension
    (16, 4, 8),
]

for TM, TN, TK in tile_sizes:
    C_tiled = gemm_tiled(A_rand, B_rand, TM, TN, TK)
    assert np.array_equal(C_tiled, C_ref), \
        f"FAILED at TM={TM}, TN={TN}, TK={TK}"
    print(f"OK  TM={TM:2d}  TN={TN:2d}  TK={TK:2d}")

print("\nAll tile sizes match the naive reference. Tiling logic verified.")

OK  TM= 4  TN= 4  TK= 4
OK  TM= 8  TN= 8  TK= 8
OK  TM=16  TN=16  TK=16
OK  TM= 4  TN= 8  TK=16
OK  TM=16  TN= 4  TK= 8

All tile sizes match the naive reference. Tiling logic verified.
